# Day 4 — LangChain and Toolchains
**GenAI 403 | Week 2**

### What we cover today
| # | Topic |
|---|-------|
| 1 | LLMChain & SimpleSequentialChain |
| 2 | Memory types (ConversationBuffer, Summary) |
| 3 | Retrievers (FAISS vector store) |
| 4 | Multi-chain RAG with memory |
| 5 | LangChain Agent with calculator + web search tools |
| 6 | Custom tool with @tool decorator |

> **Two tracks:**
> - 🔵 **Paid** — OpenAI API (`gpt-4o-mini` + `text-embedding-3-small`)
> - 🟢 **Free** — Ollama local LLM (`llama3.2`) + HuggingFace embeddings
>
> Set `USE_FREE = True` or `False` in Section 1 to switch between them.

---
## 0 — Install Dependencies
Run this cell once, then **restart the kernel** before continuing.

In [54]:
%pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-ollama \
    faiss-cpu \
    sentence-transformers \
    python-dotenv \
    duckduckgo-search \
    numexpr


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---
## 1 — Load API Key & Choose Your Track

Create a file called `.env` in the **same folder as this notebook** containing:

```
OPENAI_API_KEY=sk-your-key-here
```

Get your key at: **platform.openai.com → API Keys → Create new secret key**

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env from the current directory

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("OpenAI key loaded successfully")
else:
    print("WARNING: OPENAI_API_KEY not found — check your .env file")

OpenAI key loaded successfully


In [5]:
# ── 🔵 PAID: OpenAI ────────────────────────────────────────────────────────
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

paid_llm = ChatOpenAI(
    model="gpt-4o-mini",    # swap to "gpt-4o" for more reasoning power
    temperature=0.3,
    api_key=OPENAI_API_KEY
)

paid_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_API_KEY
)

print("OpenAI LLM + Embeddings ready")

/Users/irumzahra/Library/CloudStorage/OneDrive-Personal/Desktop/GenAI403/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OpenAI LLM + Embeddings ready


In [6]:
# ── 🟢 FREE: Ollama (local) + HuggingFace Embeddings ──────────────────────
# Setup steps (one-time):
#   1. Download and install Ollama from https://ollama.com
#   2. Open a terminal and run:  ollama pull llama3.2
#   3. Keep Ollama running in the background while using this notebook

from langchain_ollama import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings

free_llm = ChatOllama(
    model="llama3.2",
    temperature=0.3
)

free_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"  # auto-downloads ~90MB on first run
)

print("Ollama LLM + HuggingFace Embeddings ready")

/var/folders/g8/3fn287yx2y5c2h5txycsrc9h0000gn/T/ipykernel_14971/4113149686.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  free_embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1063.40it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ollama LLM + HuggingFace Embeddings ready


In [7]:
# ── Choose your track — change this ONE line to switch for the whole notebook

USE_FREE = False   # False = OpenAI (paid) | True = Ollama + HuggingFace (free)

llm        = free_llm        if USE_FREE else paid_llm
embeddings = free_embeddings if USE_FREE else paid_embeddings

track = "🟢 FREE (Ollama + HuggingFace)" if USE_FREE else "🔵 PAID (OpenAI gpt-4o-mini)"
print("Active track:", track)

Active track: 🔵 PAID (OpenAI gpt-4o-mini)


---
## 2 — LLMChain

**LLMChain** = a PromptTemplate + an LLM wired together.  
Variables inside `{}` in the template are filled in when you call `.invoke()`.

In [8]:
from langchain_core.prompts import PromptTemplate

# Step 1: define the prompt template
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in 3 bullet points. Be concise."
)

# Step 2: build the chain using pipe operator
chain = template | llm

# Step 3: run it
result = chain.invoke({"topic": "vector databases"})
print(result.content)


- **High-Dimensional Data Storage**: Vector databases are designed to efficiently store and retrieve high-dimensional vectors, which represent complex data like images, text, and audio in a numerical format.

- **Similarity Search**: They enable fast similarity searches using algorithms like k-nearest neighbors (KNN), allowing users to find items that are similar based on their vector representations.

- **Scalability and Performance**: Vector databases are optimized for handling large datasets and can scale horizontally, making them suitable for applications in machine learning, recommendation systems, and natural language processing.


---
## 3 — SimpleSequentialChain

The **output of Chain 1 automatically becomes the input of Chain 2**.  
Great for multi-step pipelines where each step builds on the previous one.

In [9]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Chain 1: summarize a concept in one paragraph
prompt_1 = PromptTemplate(
    input_variables=["concept"],
    template="Summarize the concept of '{concept}' in one short paragraph."
)

# Chain 2: turn that summary into exam questions
prompt_2 = PromptTemplate(
    input_variables=["summary"],
    template="Based on this explanation:\n{summary}\n\nWrite 2 short exam questions."
)

# Wire them: chain_1 output → chain_2 input
seq_chain = (
    prompt_1
    | llm
    | StrOutputParser()
    | (lambda summary: {"summary": summary})
    | prompt_2
    | llm
    | StrOutputParser()
)

result = seq_chain.invoke({"concept": "Retrieval-Augmented Generation"})
print("\n--- Final Output ---")
print(result)



--- Final Output ---
1. Explain the concept of Retrieval-Augmented Generation (RAG) and describe how it enhances the quality of generated responses in applications like question answering.

2. What are the two main components of the RAG approach, and how do they work together to improve the relevance and accuracy of generated text?


In [10]:
# Same as above but step by step to see intermediate output

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_1 = PromptTemplate(
    input_variables=["concept"],
    template="Summarize the concept of '{concept}' in one short paragraph."
)

prompt_2 = PromptTemplate(
    input_variables=["summary"],
    template="Based on this explanation:\n{summary}\n\nWrite 2 short exam questions."
)

# Run step by step to see intermediate output
chain_1 = prompt_1 | llm | StrOutputParser()
summary = chain_1.invoke({"concept": "Retrieval-Augmented Generation"})

print("--- Chain 1 Output (Summary) ---")
print(summary)

chain_2 = prompt_2 | llm | StrOutputParser()
result = chain_2.invoke({"summary": summary})

print("\n--- Chain 2 Output (Exam Questions) ---")
print(result)


--- Chain 1 Output (Summary) ---
Retrieval-Augmented Generation (RAG) is a hybrid approach that combines traditional information retrieval techniques with generative models to enhance the quality and relevance of generated text. In RAG, a model first retrieves relevant documents or data from a large corpus based on a given query, and then uses this retrieved information to inform and improve the generation of responses. This method allows for more accurate and contextually rich outputs by leveraging external knowledge, making it particularly effective for tasks that require up-to-date information or specialized knowledge beyond the model's training data.

--- Chain 2 Output (Exam Questions) ---
1. Explain the key components of the Retrieval-Augmented Generation (RAG) approach and discuss how it improves the quality of generated text compared to traditional generative models.

2. In what scenarios would the RAG method be particularly beneficial, and why is it important for tasks that re

---
## 4 — Memory Types

Memory lets a chain remember previous conversation turns.

| Type | Stores | Best for |
|---|---|---|
| `ConversationBufferMemory` | Every message verbatim | Short conversations |
| `ConversationSummaryMemory` | A compressed LLM summary | Long conversations (saves tokens) |


#### 4a: Buffer Memory

Three things being brought in:

`ChatPromptTemplate` — builds the structured prompt sent to the LLM

`MessagesPlaceholder` — a reserved slot inside the prompt where history gets injected

`InMemoryChatMessageHistory` — a simple list that stores past messages in RAM

`RunnableWithMessageHistory` — the wrapper that automates loading and saving history around your chain

In [23]:
# 4a: Buffer Memory → RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# This defines the shape of every message sent to the LLM.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="chat_history"), # MessagesPlaceholder is the key piece — it tells the prompt "leave this slot empty for now, history will be inserted here at runtime."
    ("human", "{input}")
])
# This chain has no memory yet, Memory is added in the next step
chain = prompt | llm

# Session store — each session_id gets its own history
store = {}

# get_session_history() is called automatically before every LLM call. 
# It does one simple thing: look up the session ID in the store and return that session's history. 
# If the session is new, it creates a fresh empty history first.
def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Wrap the Chain with Memory
# RunnableWithMessageHistory wraps any chain and automatically manages the history for you, as long as you provide a way to fetch the right history based on session_id.
"""Before the LLM call:
  1. Read session_id from config
  2. Call get_session_history(session_id) → load past messages
  3. Inject those messages into the chat_history placeholder in the prompt

After the LLM call:
  4. Save the user's input + LLM's response back into the store
  """
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# Running 3 Turns through the chain with memory. Notice how the bot's responses get more context-aware as the conversation progresses.
turns = [
    "My name is Ali.",
    "What is the capital of France?",
    "What is my name?"
]

print("=== Buffer Memory (RunnableWithMessageHistory) ===")
for turn in turns:
    print(f"User : {turn}")
    response = chain_with_history.invoke(
        {"input": turn},
        config={"configurable": {"session_id": "session_1"}}
    )
    print(f"Bot  : {response.content}\n")

"""Each .invoke() call passes two things:

{"input": turn} — the current message
config={"configurable": {"session_id": "session_1"}} — which session's history to use"""

=== Buffer Memory (RunnableWithMessageHistory) ===
User : My name is Ali.
Bot  : Nice to meet you, Ali! How can I assist you today?

User : What is the capital of France?
Bot  : The capital of France is Paris.

User : What is my name?
Bot  : Your name is Ali.



In [26]:
import json

# Convert store to a readable dict first
store_dict = {
    session_id: [
        {"role": msg.type, "content": msg.content}
        for msg in history.messages
    ]
    for session_id, history in store.items()
}

print(json.dumps(store_dict, indent=4))

{
    "session_1": [
        {
            "role": "human",
            "content": "My name is Ali."
        },
        {
            "role": "ai",
            "content": "Nice to meet you, Ali! How can I assist you today?"
        },
        {
            "role": "human",
            "content": "What is the capital of France?"
        },
        {
            "role": "ai",
            "content": "The capital of France is Paris."
        },
        {
            "role": "human",
            "content": "What is my name?"
        },
        {
            "role": "ai",
            "content": "Your name is Ali."
        }
    ]
}


#### 4b: Summary Memory 

Unlike Buffer Memory which stores every message verbatim, this approach compresses the conversation into a one-paragraph summary after each turn. 

The LLM only ever receives that summary — not the full history. Token cost stays flat no matter how long the conversation gets.

`InMemorySaver` — saves the graph's state between .invoke() calls (like a checkpoint file, but in RAM)

`StateGraph` — lets you define a workflow as a graph of nodes

`MessagesState` — a built-in state type that tracks a messages list

`START` — a special constant meaning "the entry point of the graph"

`HumanMessage` / `SystemMessage` — typed message objects (LangGraph needs typed messages, not plain strings)

In [58]:
import langchain
print(langchain.__version__)

1.2.15


In [28]:
# 4b: Summary Memory → LangGraph with InMemorySaver
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState, START
from langchain_core.messages import HumanMessage, SystemMessage

# Summarize helper
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

"""
Build the Summarizer
This is a simple summarization chain that takes the entire conversation as input and produces a concise summary. 
We will call this chain after every turn to keep our summary up to date.
"""
summarize_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize the following conversation into one concise paragraph:"),
    ("human", "{conversation}")
])

# StrOutputParser() → strips the AIMessage wrapper, returns a plain string
summarize_chain = summarize_prompt | llm | StrOutputParser() 

"""
summary_store: A plain Python dictionary with one key "text". 
It starts empty and gets overwritten after every turn with the latest compressed summary. 
This lives outside the graph so it persists across all invocations.
The Summary Store is a simple dictionary that holds the latest summary of the conversation.
"""

summary_store = {"text": ""}

"""
call_model() is the main function that gets called on every turn.
It does three things:
1. Reads the current summary from summary_store
2. Calls the LLM with the current turn + summary as context
3. Updates the summary after getting the response
"""
def call_model(state: MessagesState): # state is automatically passed in by LangGraph. It contains the full messages list for this thread.
    #  Build the prompt using the summary and the conversation history
    summary = summary_store["text"]
    messages = state["messages"]
    
    # Instead of injecting the full history, we inject just the compressed summary as a SystemMessage.
    # The LLM sees context from past turns (via the summary) plus only the current message — never the full raw history.
    system = SystemMessage(content=f"You are a helpful assistant. Conversation summary so far: {summary}")
    response = llm.invoke([system] + messages)
    """
    After the LLM responds, we immediately compress everything into a new summary.
    messages + [response] combines the current turn's messages with the fresh response, then formats them as a readable string
    That whole block of text gets sent to summarize_chain which compresses it into one paragraph — stored back into summary_store["text"] — ready for the next turn.
    """
    # Update summary after each turn
    conversation_text = "\n".join(
        f"{'User' if isinstance(m, HumanMessage) else 'Bot'}: {m.content}"
        for m in messages + [response]
    )
    summary_store["text"] = summarize_chain.invoke({"conversation": conversation_text})
    """
    LangGraph requires the node to return a dict matching the state shape. MessagesState expects {"messages": [...]}. 
    LangGraph automatically appends this response to the thread's message list.
    """
    return {"messages": [response]}


"""
Build graph
Think of this as drawing a flowchart with one box:
START ──→ [ model node = call_model() ] ──→ END
Every time we invoke the graph, it starts at START, runs call_model(), and appends the response to the messages state.

StateGraph(MessagesState) — create a graph that tracks messages as state
add_node("model", call_model) — register our function as a node named "model"
add_edge(START, "model") — whenever the graph is invoked, go straight to "model"

This graph only has one node. In more complex systems you'd have multiple nodes (retriever → model → validator etc.) connected with edges.
"""
builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")

"""
Compile with Checkpointer
compile() locks the graph structure and makes it runnable. We also pass in a checkpointer — this is what LangGraph uses to save and load state.
InMemorySaver is what gives the graph persistent state across multiple .invoke() calls. Without it, the graph would forget messages after every call.
With checkpointer:
  invoke() call 1 → state saved to InMemorySaver
  invoke() call 2 → state loaded from InMemorySaver → messages appended → saved again

Without checkpointer:
  invoke() call 1 → state lost after call ends
  invoke() call 2 → starts fresh, no history
"""
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)


turns = [
    "I am building a chatbot for a hospital.",
    "It should answer patient queries about appointments.",
    "What should I focus on to make it reliable?"
]
"""
thread_id is like session_id from Buffer Memory — it tells the checkpointer which thread's state to load. 
Different thread_id = completely separate conversation.
result['messages'][-1] — the last message in the state is always the most recent bot response.
"""

config = {"configurable": {"thread_id": "thread_1"}}

print("=== Summary Memory (LangGraph + InMemorySaver) ===")
for turn in turns:
    print(f"User : {turn}")
    result = graph.invoke({"messages": [HumanMessage(content=turn)]}, config=config)
    print(f"Bot  : {result['messages'][-1].content}\n")

print("--- Compressed Memory Buffer ---")
print(summary_store["text"])


=== Summary Memory (LangGraph + InMemorySaver) ===
User : I am building a chatbot for a hospital.
Bot  : That sounds like a great project! What specific features or functionalities are you planning to include in the hospital chatbot? For example, will it assist with appointment scheduling, provide information about services, answer common medical questions, or something else?

User : It should answer patient queries about appointments.
Bot  : That's a useful feature! For the appointment-related queries, you might consider including functionalities such as:

1. **Appointment Scheduling**: Allow patients to book, reschedule, or cancel appointments directly through the chatbot.

2. **Appointment Reminders**: Send reminders to patients about their upcoming appointments.

3. **Availability Check**: Provide information on available time slots for specific doctors or services.

4. **Location and Directions**: Offer information on where the appointments will take place, including directions to

### Buffer Memory vs Summary Memory — Side by Side

|   | BUFFER MEMORY | SUMMARY MEMORY |
|---|---|---|
| What LLM receives | Full raw history | Compressed summary + current msg |
| Token cost  | Grows every turn | Stays roughly flat |
| Detail preserved | 100% exact | Key points only |
| Best for | Short conversations | Long conversations
| Risk | Hits context limit | May lose fine details

---
## 5 — Retriever with FAISS Vector Store 

A **retriever** finds the most relevant document chunks for a query using vector similarity.  
FAISS stores everything in memory — no external database or API needed.

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Sample documents — in a real project these would come from files/databases
raw_docs = [
    "LangChain is a framework for building applications powered by language models. It supports chains, agents, and memory.",
    "FAISS (Facebook AI Similarity Search) is a library for fast vector similarity search. It stores embeddings and retrieves nearest neighbors.",
    "RAG (Retrieval-Augmented Generation) combines a retriever and a generator. It grounds LLM responses in real documents, reducing hallucinations.",
    "Memory in LangChain allows conversational chains to track context across turns. Types include buffer memory, summary memory, and entity memory.",
    "Agents use tools to take actions. A LangChain agent can call a calculator, search the web, or query a database based on the user request."
]

# Wrap strings as Document objects
docs = [Document(page_content=text) for text in raw_docs]

# Split into chunks (these are short, so each stays as one chunk)
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(docs)

# Build FAISS vector store: embed each chunk and store
vector_store = FAISS.from_documents(chunks, embeddings)

# Create a retriever — k=2 means return top 2 most similar chunks
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Test retrieval
query = "How does RAG reduce hallucinations?"
results = retriever.invoke(query)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i}: {doc.page_content}")


Query: How does RAG reduce hallucinations?

Result 1: RAG (Retrieval-Augmented Generation) combines a retriever and a generator. It grounds LLM responses in real documents, reducing hallucinations.
Result 2: Memory in LangChain allows conversational chains to track context across turns. Types include buffer memory, summary memory, and entity memory.


---
## 6 — Multi-chain RAG with Memory

combines three things:
1. **Retriever** → fetches relevant chunks from the vector store
2. **Memory** → tracks the full conversation history
3. **LLM** → generates an answer grounded in retrieved context

This is the core pattern of a production RAG chatbot.

In [31]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState, START
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Condense follow-up question using chat history
condense_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given the chat history and a follow-up question, rephrase it as a standalone question. If it's already standalone, return it as-is."),
    ("human", "Chat history:\n{history}\n\nFollow-up question: {question}")
])
condense_chain = condense_prompt | llm | StrOutputParser()

# Answer using retrieved context
answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the question using only the context below.\n\nContext: {context}"),
    ("human", "{question}")
])
answer_chain = answer_prompt | llm | StrOutputParser()

def call_model(state: MessagesState):
    messages = state["messages"]
    
    # Extract history (all but last message)
    history = "\n".join(
        f"{'User' if isinstance(m, HumanMessage) else 'Bot'}: {m.content}"
        for m in messages[:-1]
    )
    question = messages[-1].content

    # Condense follow-up into standalone question
    standalone = condense_chain.invoke({"history": history, "question": question})

    # Retrieve relevant docs
    docs = retriever.invoke(standalone)
    context = "\n".join(doc.page_content for doc in docs)

    # Generate answer
    answer = answer_chain.invoke({"question": standalone, "context": context})

    from langchain_core.messages import AIMessage
    return {"messages": [AIMessage(content=answer)]}

# Build graph with InMemorySaver for buffer memory
builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_edge(START, "model")

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

# Multi-turn conversation
# Notice Q2 uses 'it' to refer to FAISS from Q1 — memory resolves the reference
questions = [
    "What is FAISS?",
    "How does it relate to RAG?",
    "What memory types does LangChain support? in latest version, explain each and implementation"
]

config = {"configurable": {"thread_id": "rag_session_1"}}

for question in questions:
    print(f"\nQ: {question}")
    result = graph.invoke({"messages": [HumanMessage(content=question)]}, config=config)
    print(f"A: {result['messages'][-1].content}")



Q: What is FAISS?
A: FAISS (Facebook AI Similarity Search) is a library for fast vector similarity search that stores embeddings and retrieves nearest neighbors.

Q: How does it relate to RAG?
A: FAISS is related to RAG in that it can be used as the retriever component within the RAG framework. FAISS enables fast vector similarity search to retrieve relevant embeddings, which can then be used to ground the responses of the generator in real documents, thereby enhancing the accuracy and reliability of the generated content.

Q: What memory types does LangChain support? in latest version, explain each and implementation
A: LangChain supports three types of memory: buffer memory, summary memory, and entity memory. 

1. **Buffer Memory**: This type of memory allows the conversational chain to keep track of the entire conversation history. It stores each interaction in a sequential manner, enabling the model to reference previous turns directly. Implementation typically involves appending 

---
## 7 — LangChain Agent with Tools

An **agent** reads the user query and decides which tool to call.  
It follows a Reason → Act loop until it has a final answer.

| Tool | What it does | Cost |
|---|---|---|
| `llm-math` | Evaluates math expressions | Free (runs locally) |
| `ddg-search` | Web search via DuckDuckGo | Free (no API key) |

In [39]:
%pip install -U ddgs


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [40]:
# from langchain.agents import initialize_agent, load_tools, AgentType
from langgraph.prebuilt import create_react_agent
# from langchain_community.agent_toolkits import load_tools
from langchain_community.agent_toolkits.load_tools import load_tools



# Load built-in tools
tools = load_tools(
    ["llm-math", "ddg-search"],
    llm=llm    # llm-math needs the LLM to parse the math expression
)

agent = create_react_agent(llm, tools)
result = agent.invoke({"messages": [{"role": "user", "content": "What is 17 raised to the power of 3, divided by 4.9?"}]})
# Access the final answer:
print(result["messages"][-1].content)



print("Agent ready. Tools:", [t.name for t in tools])

/var/folders/g8/3fn287yx2y5c2h5txycsrc9h0000gn/T/ipykernel_14971/6933915.py:14: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


17 raised to the power of 3, divided by 4.9, is approximately 1002.65.
Agent ready. Tools: ['Calculator', 'duckduckgo_search']


In [43]:
# Test 1: Math — agent should pick llm-math
result = agent.invoke({"messages": [{"role": "user", "content": "What is 17 raised to the power of 3, divided by 4.9?"}]})
print("\nFinal Answer:", result["messages"])


Final Answer: [HumanMessage(content='What is 17 raised to the power of 3, divided by 4.9?', additional_kwargs={}, response_metadata={}, id='1de00e62-1104-4176-b48c-9b3019b91cc5'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 123, 'total_tokens': 148, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DWVPcesGl0p4j6MX2IH4ovRYTCNkl', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019da7f9-23c8-7503-9461-db7ce13c3537-0', tool_calls=[{'name': 'Calculator', 'args': {'__arg1': '(17^3) / 4.9'}, 'id': 'call_QuIctv4e32kw2bx90NqaOAYH', 'type': 'tool_call'}], invalid_tool_calls=[], usage_meta

In [46]:
# Test 2: Web search — agent should pick ddg-search
result = agent.invoke({"messages": [{"role": "user", "content": "Who founded LangChain and when was it first released?"}]})
print("\nFinal Answer:", result["messages"][-1].content)


Final Answer: LangChain was founded by Harrison Chase in October 2022. The first commit to the LangChain repository was made on October 24, 2022, marking its initial release as a lightweight Python wrapper for prompt templates and basic chain composition.


In [47]:
# Test 3: Multi-step — agent chains search → math
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Search for the current population of Pakistan, then calculate 10% of that number."}]}
)
print("\nFinal Answer:", result["messages"][-1].content)


Final Answer: The current population of Pakistan is approximately 241,499,431. Ten percent of that number is 24,149,943.1.


In [48]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Search for the current population of Pakistan, then calculate 10% of that number."}]}
)

# Show all messages to see tool calls
for msg in result["messages"]:
    print(f"\n[{msg.__class__.__name__}]")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool called: {tc['name']}")
            print(f"  Input: {tc['args']}")
    elif hasattr(msg, "name") and msg.name:   # ToolMessage
        print(f"  Tool result from: {msg.name}")
        print(f"  Output: {msg.content[:200]}")
    else:
        print(f"  {msg.content[:300]}")

print("\nFinal Answer:", result["messages"][-1].content)



[HumanMessage]
  Search for the current population of Pakistan, then calculate 10% of that number.

[AIMessage]
  Tool called: duckduckgo_search
  Input: {'query': 'current population of Pakistan 2023'}

[ToolMessage]
  Tool result from: duckduckgo_search
  Output: The 2023 Census of Pakistan was the seventh national census and detailed enumeration of the Pakistani population. ... of Pakistan requires that a ... The current population of Pakistan is 257,916,617 

[AIMessage]
  Tool called: Calculator
  Input: {'__arg1': '241490000 * 0.10'}

[ToolMessage]
  Tool result from: Calculator
  Output: Answer: 24149000.0

[AIMessage]
  The current population of Pakistan is approximately 241.49 million. Ten percent of that number is 24,149,000.

Final Answer: The current population of Pakistan is approximately 241.49 million. Ten percent of that number is 24,149,000.


---
## 8 — Custom Tool with @tool

Any Python function can be turned into a LangChain tool with `@tool`.  
The **docstring** is critical — the agent reads it to decide when to use the tool.

In [49]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def word_count(text: str) -> str:
    """Counts the number of words in the given text. Input should be a plain string."""
    count = len(text.split())
    return f"The text has {count} words."

@tool
def reverse_text(text: str) -> str:
    """Reverses the characters in the given text string. Input should be a plain string."""
    return f"Reversed: {text[::-1]}"

# Build agent with only our custom tools
custom_agent = create_react_agent(llm, tools=[word_count, reverse_text])

# Test 1: word count
result = custom_agent.invoke(
    {"messages": [{"role": "user", "content": "How many words are in: 'LangChain makes building LLM apps easy'?"}]}
)
print("\nFinal Answer:", result["messages"][-1].content)


/var/folders/g8/3fn287yx2y5c2h5txycsrc9h0000gn/T/ipykernel_14971/213197775.py:16: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  custom_agent = create_react_agent(llm, tools=[word_count, reverse_text])



Final Answer: The text "LangChain makes building LLM apps easy" has 6 words.


In [51]:
result = custom_agent.invoke(
    {"messages": [{"role": "user", "content": "How many words are in: 'LangChain makes building LLM apps easy'?"}]}
)

# Show all messages to see tool calls
for msg in result["messages"]:
    print(f"\n[{msg.__class__.__name__}]")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool called: {tc['name']}")
            print(f"  Input: {tc['args']}")
    elif hasattr(msg, "name") and msg.name:   # ToolMessage
        print(f"  Tool result from: {msg.name}")
        print(f"  Output: {msg.content[:200]}")
    else:
        print(f"  {msg.content[:300]}")

print("\nFinal Answer:", result["messages"][-1].content)



[HumanMessage]
  How many words are in: 'LangChain makes building LLM apps easy'?

[AIMessage]
  Tool called: word_count
  Input: {'text': 'LangChain makes building LLM apps easy'}

[ToolMessage]
  Tool result from: word_count
  Output: The text has 6 words.

[AIMessage]
  The text "LangChain makes building LLM apps easy" has 6 words.

Final Answer: The text "LangChain makes building LLM apps easy" has 6 words.


In [52]:
result = custom_agent.invoke(
    {"messages": [{"role": "user", "content": "Reverse the text: 'hello world'?"}]}
)

# Show all messages to see tool calls
for msg in result["messages"]:
    print(f"\n[{msg.__class__.__name__}]")
    if hasattr(msg, "tool_calls") and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"  Tool called: {tc['name']}")
            print(f"  Input: {tc['args']}")
    elif hasattr(msg, "name") and msg.name:   # ToolMessage
        print(f"  Tool result from: {msg.name}")
        print(f"  Output: {msg.content[:200]}")
    else:
        print(f"  {msg.content[:300]}")

print("\nFinal Answer:", result["messages"][-1].content)



[HumanMessage]
  Reverse the text: 'hello world'?

[AIMessage]
  Tool called: reverse_text
  Input: {'text': 'hello world'}

[ToolMessage]
  Tool result from: reverse_text
  Output: Reversed: dlrow olleh

[AIMessage]
  The reversed text is: **dlrow olleh**.

Final Answer: The reversed text is: **dlrow olleh**.


---
## Summary

| Concept | Key Class / Function |
|---|---|
| **LLMChain** | `PromptTemplate` + `LLMChain` |
| **Sequential Chain** | `SimpleSequentialChain` |
| **Buffer Memory** |  stores full history |
| **Summary Memory** |  compresses history |
| **Retriever** | `FAISS.from_documents()` + `.as_retriever()` |
| **RAG + Memory** | `ConversationalRetrievalChain` |
| **Agent + Tools** | `initialize_agent` with `llm-math`, `ddg-search` |
| **Custom Tool** | `@tool` decorator on any Python function |

### Free vs Paid
| Component | 🔵 Paid | 🟢 Free |
|---|---|---|
| LLM | OpenAI `gpt-4o-mini` | Ollama `llama3.2` (runs locally) |
| Embeddings | OpenAI `text-embedding-3-small` | HuggingFace `all-MiniLM-L6-v2` |
| Web Search | DuckDuckGo (free in both tracks) | DuckDuckGo (free in both tracks) |
| Vector Store | FAISS (free in both tracks) | FAISS (free in both tracks) |

> **Next session:** Day 5 — LLM Evaluation, Guardrails & Hallucination Testing